# TP : prédire l'espèce d'un iris avec scikit-learn

Ce notebook présente un projet complet de **classification** : exploration d'un CSV public, séparation des données, entraînement, évaluation et prédiction d'une nouvelle fleur.

Exécutez les cellules dans l'ordre dans Google Colab. Le seul accès externe nécessaire est le téléchargement du CSV public via son URL.


## 1. Importer les bibliothèques

`pandas` manipule le tableau, `matplotlib` produit les graphiques et `scikit-learn` fournit les outils de machine learning. Aucun paquet supplémentaire n'est à installer dans Colab.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


## 2. Charger et examiner le CSV

Le fichier [iris.csv](https://github.com/mwaskom/seaborn-data/blob/master/iris.csv) est public. Ses quatre premières colonnes mesurent les sépales et les pétales en centimètres ; `species` indique l'espèce. L'URL brute est lisible directement par `pd.read_csv`.


In [ ]:
# Chargement direct depuis une URL publique : aucun compte ni clé API.
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
df = pd.read_csv(url)

print("Premières lignes :")
print(df.head())
print("\nDimensions (lignes, colonnes) :", df.shape)
print("\nNoms des colonnes :", df.columns.tolist())
print("\nTypes des colonnes :")
print(df.dtypes)


## 3. Explorer les données

Les statistiques résument les mesures. Le nombre de valeurs manquantes montre si une correction est nécessaire. La distribution de `species` révèle l'équilibre des classes. Les graphiques montrent le nombre de fleurs par espèce et le lien entre deux mesures des pétales.


In [ ]:
# Statistiques descriptives des variables numériques.
print("Statistiques descriptives :")
print(df.describe())

# Le comptage est fait colonne par colonne, cible comprise.
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

print("\nNombre de fleurs par espèce :")
print(df["species"].value_counts())

# Deux visualisations simples, sans bibliothèque graphique supplémentaire.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["species"].value_counts().sort_index().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Distribution de la cible")
axes[0].set_xlabel("Espèce")
axes[0].set_ylabel("Nombre de fleurs")
axes[0].tick_params(axis="x", rotation=0)

for espece, groupe in df.groupby("species"):
    axes[1].scatter(
        groupe["petal_length"], groupe["petal_width"], label=espece, alpha=0.75
    )
axes[1].set_title("Mesures des pétales")
axes[1].set_xlabel("Longueur (cm)")
axes[1].set_ylabel("Largeur (cm)")
axes[1].legend()
plt.tight_layout()
plt.show()


## 4. Définir les caractéristiques `X` et la cible `y`

`X` contient les quatre mesures numériques utilisées pour prédire. `y` contient l'espèce à prédire. Les caractéristiques décrivent une fleur ; la cible est la réponse attendue pendant l'apprentissage.

Ce CSV ne contient normalement aucune valeur manquante dans les mesures et aucune caractéristique catégorielle. Le pipeline ci-dessous garde toutefois une imputation médiane, qui permet de traiter une mesure manquante si elle apparaît plus tard. Aucun encodage n'est requis. La normalisation est utile à la régression logistique ; elle sera apprise sur les données d'entraînement seulement.


In [ ]:
# L'ordre explicite des colonnes sera repris pour la nouvelle fleur.
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = df[features]
y = df["species"]

print("Caractéristiques X :", X.columns.tolist())
print("Cible y :", y.name)
print("Valeurs manquantes dans X :", int(X.isna().sum().sum()))


## 5. Séparer l'entraînement et le test

L'ensemble d'entraînement sert à ajuster le modèle. L'ensemble de test est mis de côté pour mesurer ses performances sur des fleurs qu'il n'a pas vues pendant l'entraînement. `test_size=0.2` réserve 20 % des lignes au test. `random_state=42` rend ce partage reproductible. `stratify=y` conserve les proportions des trois espèces dans chaque ensemble.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Nombre de fleurs pour l'entraînement :", len(X_train))
print("Nombre de fleurs pour le test :", len(X_test))
print("Espèces dans le test :")
print(y_test.value_counts())


## 6. Construire et entraîner le modèle

La **régression logistique** est un algorithme simple pour distinguer plusieurs classes à partir de mesures numériques. Malgré son nom, elle effectue ici une classification.

Le pipeline applique successivement une éventuelle imputation, une normalisation, puis la régression logistique. `fit` apprend ces transformations et le modèle uniquement sur l'ensemble d'entraînement. Cela évite de faire intervenir les données de test dans la préparation.


In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# Le pipeline ajuste l'imputation, l'échelle et le classifieur en une seule étape.
model.fit(X_train, y_train)
print("Modèle entraîné. Espèces possibles :", model.classes_.tolist())


## 7. Prédire les fleurs du test

`predict` renvoie une espèce pour chaque fleur du test. Le tableau suivant rapproche quelques prédictions des réponses réelles.


In [ ]:
y_pred = model.predict(X_test)

comparaison = pd.DataFrame({
    "espèce_réelle": y_test.to_numpy(),
    "espèce_prédite": y_pred,
})
print(comparaison.head(10).to_string(index=False))


## 8. Évaluer les prédictions

L'**accuracy** est la part de prédictions correctes. La **précision** indique, pour une espèce prédite, la part de prédictions justes. Le **rappel** mesure la part de fleurs de cette espèce retrouvées. Le **F1-score** combine précision et rappel. Comme il y a trois espèces, `average="macro"` calcule la moyenne des scores de chaque espèce. La matrice de confusion détaille les bonnes réponses sur la diagonale et les erreurs ailleurs.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
erreurs = int(np.sum(y_test.to_numpy() != y_pred))

print(f"Accuracy : {accuracy:.3f}")
print(f"Précision macro : {precision:.3f}")
print(f"Rappel macro : {recall:.3f}")
print(f"F1-score macro : {f1:.3f}")
print(f"Erreurs sur {len(y_test)} fleurs de test : {erreurs}")
print(f"Interprétation : {accuracy:.1%} des fleurs du test sont correctement classées.")

matrice = confusion_matrix(y_test, y_pred, labels=model.classes_)
print("\nMatrice de confusion (lignes = réalité, colonnes = prédiction) :")
print(pd.DataFrame(matrice, index=model.classes_, columns=model.classes_))
ConfusionMatrixDisplay(matrice, display_labels=model.classes_).plot(cmap="Blues")
plt.title("Matrice de confusion sur le test")
plt.show()


## 9. Vérifier la stabilité par validation croisée

La validation croisée divise **l'ensemble d'entraînement** en cinq parties. À chaque tour, le pipeline est réappris sur quatre parties et évalué sur la cinquième. Les scores donnent une idée de la stabilité du modèle selon le partage des données. L'ensemble de test reste réservé à l'évaluation précédente ; il n'entre pas dans cette validation croisée.


In [ ]:
# Le mélange reproductible évite que les plis suivent l'ordre du CSV.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_cv = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")

print("Accuracy de chaque pli :", np.round(scores_cv, 3))
print(f"Accuracy moyenne : {scores_cv.mean():.3f}")
print(f"Écart-type des scores : {scores_cv.std():.3f}")
print("Un écart-type faible suggère des performances assez stables entre les plis.")


## 10. Utiliser le modèle sur une nouvelle fleur

Voici un exemple de fleur mesurée en centimètres. Le tableau possède **exactement les quatre colonnes de `X`, dans le même ordre**. Le pipeline entraîné applique automatiquement l'imputation et la normalisation avant de prédire l'espèce. Le résultat est une estimation du modèle, pas une identification certaine.


In [ ]:
# Nouvelle observation fictive : les valeurs ne sont pas la cible à prédire.
nouvelle_donnee = pd.DataFrame(
    [[5.0, 3.4, 1.5, 0.25]],
    columns=features,
)

prediction = model.predict(nouvelle_donnee)
print("Mesures de la nouvelle fleur :")
print(nouvelle_donnee.to_string(index=False))
print("Prédiction :", prediction[0])
print(f"Le modèle classe cette fleur dans l'espèce « {prediction[0]} ».")
